# 🗓️ Conflict Resolver — SFT + GRPO Training
**Runtime: T4 GPU** | Full pipeline: Install → Baseline → SFT → GRPO → Compare

In [ ]:
# Cell 1: Install everything
!pip install -q unsloth trl datasets pydantic fastapi huggingface_hub nest_asyncio
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# Cell 2: Download environment + setup
from huggingface_hub import snapshot_download
import sys, os, asyncio, json
import nest_asyncio
nest_asyncio.apply()

env_path = snapshot_download(
    repo_id="srivtx/openenv-conflict-resolver-v2",
    repo_type="space", local_dir="./env_code"
)
sys.path.insert(0, os.path.join(env_path, "src"))

from assistant_conflict_env.environment import PersonalAssistantConflictEnv
from assistant_conflict_env.models import ConflictAction, ActionIntent, Owner, Priority

TASK_IDS = ["easy_evening_planner", "medium_multi_party_negotiation", "hard_cascade_replanning"]

# Quick test
env = PersonalAssistantConflictEnv()
r = await env.reset(task_name="easy_evening_planner")
print(f"Environment loaded! First conflict: {r.observation.current_conflict.summary}")

In [ ]:
# Cell 3: Helper functions

def heuristic_action(obs):
    c = obs.current_conflict
    if c is None:
        return ConflictAction(intent="route_message", owner="self", priority="normal", message_template="Default.")
    text = f"{c.summary} {' '.join(c.constraints)}".lower()
    if any(t in text for t in ["missing", "unclear", "timezone", "attachment"]):
        return ConflictAction(intent="ask_clarification", owner="work", priority="high", needs_clarification=True, message_template="Need clarification before execution.")
    elif any(t in text for t in ["overlap", "delay", "reservation", "check-in", "driver"]):
        return ConflictAction(intent="reschedule_event", owner="travel", priority="high", proposed_slot="after 20:30", message_template="Reschedule to protect constraints.")
    elif any(t in text for t in ["pickup", "gift", "cancel window"]):
        return ConflictAction(intent="delegate_task", owner="family", priority="normal", message_template="Delegate with confirmation.")
    elif any(t in text for t in ["payment", "renewal", "insurance"]):
        return ConflictAction(intent="route_message", owner="finance", priority="urgent", message_template="Route payment urgently.")
    elif any(t in text for t in ["final", "consolidated", "itinerary"]):
        return ConflictAction(intent="finalize_itinerary", owner="self", priority="high", message_template="Finalize timeline with risks.")
    return ConflictAction(intent="route_message", owner="self", priority="normal", message_template="Routing with fallback.")

def parse_json(text):
    text = (text or "").strip()
    try: return json.loads(text)
    except: pass
    l, r = text.find("{"), text.rfind("}")
    if l >= 0 and r > l:
        try: return json.loads(text[l:r+1])
        except: pass
    return None

async def eval_model(mdl, tok, label):
    env = PersonalAssistantConflictEnv()
    results = {}
    for tid in TASK_IDS:
        r = await env.reset(task_name=tid)
        while not r.done and r.observation.current_conflict:
            c = r.observation.current_conflict
            prompt = (
                "Return ONLY a JSON object, nothing else.\n"
                '{"intent": "...", "owner": "...", "priority": "...", '
                '"proposed_slot": "...", "needs_clarification": false, '
                '"message_template": "..."}\n\n'
                f"Conflict: {c.summary}\n"
                f"Constraints: {', '.join(c.constraints)}\n"
                "Intents: route_message/propose_plan/reschedule_event/delegate_task/ask_clarification/finalize_itinerary\n"
                "Owners: self/work/family/travel/finance/legal\n"
                "Priorities: low/normal/high/urgent\nJSON:"
            )
            msgs = [{"role": "user", "content": prompt}]
            inp = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
            out = mdl.generate(inp, max_new_tokens=200, temperature=0.05, do_sample=True, repetition_penalty=1.1)
            text = tok.decode(out[0][inp.shape[1]:], skip_special_tokens=True)
            p = parse_json(text)
            if p:
                try:
                    action = ConflictAction(
                        intent=str(p.get("intent","route_message")).lower(),
                        owner=str(p.get("owner","self")).lower(),
                        priority=str(p.get("priority","normal")).lower(),
                        proposed_slot=str(p.get("proposed_slot",""))[:80],
                        needs_clarification=bool(p.get("needs_clarification",False)),
                        message_template=str(p.get("message_template","Act."))[:500]
                    )
                except: action = heuristic_action(r.observation)
            else: action = heuristic_action(r.observation)
            r = await env.step(action)
        s = await env.state()
        score = float(s.final_score or s.average_step_score)
        results[tid] = round(score, 4)
        print(f"  [{label}] {tid}: {score:.4f}")
    return results

print("Helpers ready!")

In [ ]:
# Cell 4: Load fresh Qwen 3B
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=608, load_in_4bit=True
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0, use_gradient_checkpointing="unsloth"
)
print("Model loaded!")

In [ ]:
# Cell 5: Evaluate UNTRAINED baseline
FastLanguageModel.for_inference(model)
print("Evaluating untrained 3B...")
pretrain_scores = await eval_model(model, tokenizer, "Untrained")
avg_pre = sum(pretrain_scores.values()) / 3
print(f"\nUntrained average: {avg_pre:.4f}")

In [ ]:
# Cell 6: Build SFT dataset from correct answers
sft_data = []
for tid in TASK_IDS:
    env = PersonalAssistantConflictEnv()
    r = await env.reset(task_name=tid)
    while not r.done and r.observation.current_conflict:
        c = r.observation.current_conflict
        exp = c.expected
        prompt = (
            "Return ONLY a JSON object, nothing else.\n"
            '{"intent": "...", "owner": "...", "priority": "...", '
            '"proposed_slot": "...", "needs_clarification": false, '
            '"message_template": "..."}\n\n'
            f"Conflict: {c.summary}\n"
            f"Constraints: {', '.join(c.constraints)}\n"
            "Intents: route_message/propose_plan/reschedule_event/delegate_task/ask_clarification/finalize_itinerary\n"
            "Owners: self/work/family/travel/finance/legal\n"
            "Priorities: low/normal/high/urgent\nJSON:"
        )
        intent_val = exp.intent.value if hasattr(exp.intent, 'value') else str(exp.intent)
        owner_val = exp.owner.value if hasattr(exp.owner, 'value') else str(exp.owner)
        priority_val = exp.priority.value if hasattr(exp.priority, 'value') else str(exp.priority)
        keywords = exp.required_keywords if exp.required_keywords else [intent_val]
        answer = json.dumps({
            "intent": intent_val,
            "owner": owner_val,
            "priority": priority_val,
            "proposed_slot": exp.expected_slot_hint or "",
            "needs_clarification": exp.block_if_missing_context,
            "message_template": f"{intent_val.replace('_',' ')}: {', '.join(keywords)}. Handle with care."
        })
        text = tokenizer.apply_chat_template([
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": answer}
        ], tokenize=False)
        sft_data.append({"text": text})
        r = await env.step(heuristic_action(r.observation))

# Repeat data 15x for more training signal
sft_data = sft_data * 15
print(f"SFT dataset: {len(sft_data)} examples ({len(sft_data)//15} unique × 15 repeats)")
print(f"\nExample answer:\n{json.loads(json.dumps(sft_data[0]))}")

In [ ]:
        prompt = (
            "Return ONLY a JSON object, nothing else.\n"
            '{"intent": "...", "owner": "...", "priority": "...", '
            '"proposed_slot": "...", "needs_clarification": false, '
            '"message_template": "..."}\n\n'
            f"Conflict: {c.summary}\n"
            f"Constraints: {', '.join(c.constraints)}\n"
            "Intents: route_message/propose_plan/reschedule_event/delegate_task/ask_clarification/finalize_itinerary\n"
            "Owners: self/work/family/travel/finance/legal\n"
            "Priorities: low/normal/high/urgent\nJSON:"
        )
        intent_val = exp.intent.value if hasattr(exp.intent, 'value') else str(exp.intent)
        owner_val = exp.owner.value if hasattr(exp.owner, 'value') else str(exp.owner)
        priority_val = exp.priority.value if hasattr(exp.priority, 'value') else str(exp.priority)
        keywords = exp.required_keywords if exp.required_keywords else [intent_val]
        answer = json.dumps({
            "intent": intent_val,
            "owner": owner_val,
            "priority": priority_val,
            "proposed_slot": exp.expected_slot_hint or "",
            "needs_clarification": exp.block_if_missing_context,
            "message_template": f"{intent_val.replace('_',' ')}: {', '.join(keywords)}. Handle with care."
        })
        text = tokenizer.apply_chat_template([
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": answer}
        ], tokenize=False)
        sft_data.append({"text": text})
        r = await env.step(heuristic_action(r.observation))

# Repeat data 15x for more training signal
sft_data = sft_data * 15
print(f"SFT dataset: {len(sft_data)} examples ({len(sft_data)//15} unique × 15 repeats)")
print(f"\nExample answer:\n{json.loads(json.dumps(sft_data[0]))}")

In [ ]:
# Cell 7: SFT Training — teach correct JSON format (~5-10 min)
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

sft_dataset = Dataset.from_list(sft_data)

sft_trainer = SFTTrainer(
    model=model,
    train_dataset=sft_dataset,
    args=SFTConfig(
        output_dir="./sft_output",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        max_seq_length=608,
        logging_steps=10,
        warmup_steps=5,
        seed=42,
    ),
    processing_class=tokenizer,
)

print("SFT training (teaching correct answers)...")
sft_trainer.train()
print("SFT complete!")

In [ ]:
# Cell 8: Evaluate AFTER SFT
FastLanguageModel.for_inference(model)
print("Evaluating SFT-trained model...")
sft_scores = await eval_model(model, tokenizer, "SFT")
avg_sft = sum(sft_scores.values()) / 3
print(f"\nSFT average: {avg_sft:.4f} (was {avg_pre:.4f})")

In [ ]:
# Cell 9: GRPO Training on top of SFT (~30-40 min)
from trl import GRPOConfig, GRPOTrainer

# Collect prompts for GRPO
async def collect_prompts(episodes=30):
    env = PersonalAssistantConflictEnv()
    rows = []
    for ep in range(episodes):
        tid = TASK_IDS[ep % len(TASK_IDS)]
        r = await env.reset(task_name=tid)
        hist = []
        while not r.done and r.observation.current_conflict:
            c = r.observation.current_conflict
            prompt = (
                "Return ONLY a JSON object, nothing else.\n"
                '{"intent": "...", "owner": "...", "priority": "...", '
                '"proposed_slot": "...", "needs_clarification": false, '
                '"message_template": "..."}\n\n'
                f"Conflict: {c.summary}\n"
                f"Constraints: {', '.join(c.constraints)}\n"
                "Intents: route_message/propose_plan/reschedule_event/delegate_task/ask_clarification/finalize_itinerary\n"
                "Owners: self/work/family/travel/finance/legal\n"
                "Priorities: low/normal/high/urgent\nJSON:"
            )
            rows.append({"prompt": prompt, "task_name": tid, "history": [dict(h) for h in hist], "step": r.observation.step_index})
            action = heuristic_action(r.observation)
            r = await env.step(action)
            hist.append(action.model_dump(mode="json"))
            if len(rows) >= 180: break
        if len(rows) >= 180: break
    return rows

prompt_rows = await collect_prompts()

def reward_fn(prompts, completions, **kwargs):
    rewards = []
    for i, comp in enumerate(completions):
        text = comp[0]["content"] if isinstance(comp, list) else str(comp)
        p = parse_json(text)
        if p is None:
            rewards.append(0.0); continue
        try:
            action = ConflictAction(
                intent=str(p.get("intent","route_message")).lower(),
                owner=str(p.get("owner","self")).lower(),
                priority=str(p.get("priority","normal")).lower(),
                proposed_slot=str(p.get("proposed_slot",""))[:80],
                needs_clarification=bool(p.get("needs_clarification",False)),
                message_template=str(p.get("message_template","Act."))[:500]
            )
            row = prompt_rows[min(i, len(prompt_rows)-1)]
            env_l = PersonalAssistantConflictEnv()
            res = asyncio.run(env_l.reset(task_name=row["task_name"]))
            for h in row.get("history", []):
                if res.done: break
                res = asyncio.run(env_l.step(ConflictAction.model_validate(h)))
            if not res.done:
                res = asyncio.run(env_l.step(action))
                rewards.append(float(res.reward))
            else: rewards.append(0.0)
        except: rewards.append(0.0)
    return rewards

grpo_dataset = Dataset.from_list(prompt_rows)

grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=GRPOConfig(
        output_dir="./grpo_out",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=1e-5,
        num_train_epochs=1,
        logging_steps=5,
        max_prompt_length=512,
        max_completion_length=96,
        seed=42,
    ),
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
)

print("GRPO training (1 epoch, conservative LR)...")
grpo_trainer.train()
print("GRPO complete!")

In [ ]:
# Cell 10: Save final model
model.save_pretrained("./trained_conflict_resolver")
tokenizer.save_pretrained("./trained_conflict_resolver")
print("Model saved!")

In [ ]:
# Cell 11: Evaluate FINAL trained model
FastLanguageModel.for_inference(model)
print("Evaluating final model (SFT + GRPO)...")
final_scores = await eval_model(model, tokenizer, "Final")

In [ ]:
# Cell 12: FINAL COMPARISON TABLE
print("\n" + "="*75)
print(f"{'Task':<40} {'Before':>9} {'SFT':>9} {'SFT+GRPO':>9}")
print("="*75)
for t in TASK_IDS:
    b = pretrain_scores[t]
    s = sft_scores[t]
    f = final_scores[t]
    print(f"{t:<40} {b:>9.4f} {s:>9.4f} {f:>9.4f}")
avg_b = sum(pretrain_scores.values()) / 3
avg_s = sum(sft_scores.values()) / 3
avg_f = sum(final_scores.values()) / 3
print("="*75)
print(f"{'AVERAGE':<40} {avg_b:>9.4f} {avg_s:>9.4f} {avg_f:>9.4f}")
print(f"\n{'Improvement over baseline:':<40} {'':>9} {avg_s-avg_b:>+8.4f} {avg_f-avg_b:>+8.4f}")
print("\n🎯 Training Pipeline: Untrained → SFT (learn format) → GRPO (optimize rewards)")